# Reasoning Traces Analysis - Multi-Model Comparison

Compare reasoning trace lengths and prediction results across multiple models

## 1. Import Required Libraries

In [6]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from transformers import AutoTokenizer

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

## 2. Load Model Results Data

In [8]:
# Load results from all available models
base_dir = Path("../data/reasoning_traces")
model_dirs = sorted([d for d in base_dir.iterdir() if d.is_dir()])

print(f"Found {len(model_dirs)} models:")
for d in model_dirs:
    print(f"  - {d.name}")

# Load data for each model
models_data = {}
for model_dir in model_dirs:
    model_name = model_dir.name
    parsed_file = model_dir / "parsed_results.json"
    
    if parsed_file.exists():
        with open(parsed_file, "r") as f:
            results = json.load(f)
        models_data[model_name] = pd.DataFrame(results)
        print(f"\n{model_name}: {len(results)} samples")
        if len(results) > 0:
            print(f"  Columns: {list(results[0].keys())}")

print(f"\nTotal models with data: {len(models_data)}")

Found 2 models:
  - RedHatAI-Qwen3-32B-quantized.w4a16
  - RedHatAI-gemma-3-27b-it-quantized.w4a16

RedHatAI-Qwen3-32B-quantized.w4a16: 20 samples
  Columns: ['wildguard_id', 'prompt', 'response', 'intent', 'condition', 'ground_truth', 'predicted', 'reasoning']

RedHatAI-gemma-3-27b-it-quantized.w4a16: 20 samples
  Columns: ['wildguard_id', 'prompt', 'response', 'intent', 'condition', 'ground_truth', 'predicted', 'reasoning']

Total models with data: 2


## 3. Calculate Token Counts

In [ ]:
# Load tokenizer and compute token counts for reasoning
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")

for model_name, df in models_data.items():
    df["reasoning_tokens"] = df["reasoning"].apply(
        lambda x: len(tokenizer.encode(x)) if isinstance(x, str) else 0
    )
    print(f"\n{model_name} - Reasoning Token Stats:")
    print(f"  Mean: {df['reasoning_tokens'].mean():.0f}")
    print(f"  Median: {df['reasoning_tokens'].median():.0f}")
    print(f"  Min-Max: {df['reasoning_tokens'].min()} - {df['reasoning_tokens'].max()}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


RedHatAI-Qwen3-32B-quantized.w4a16 - Reasoning Token Stats:
  Mean: 129
  Median: 128
  Min-Max: 92 - 172

RedHatAI-gemma-3-27b-it-quantized.w4a16 - Reasoning Token Stats:
  Mean: 108
  Median: 104
  Min-Max: 63 - 170


## 4. Summary Statistics

In [11]:
# Generate summary statistics
summary_stats = []

for model_name, df in models_data.items():
    summary_stats.append({
        "Model": model_name,
        "Total Samples": len(df),
        "Avg Tokens": f"{df['reasoning_tokens'].mean():.0f}",
        "Median Tokens": f"{df['reasoning_tokens'].median():.0f}",
        "Min Tokens": df["reasoning_tokens"].min(),
        "Max Tokens": df["reasoning_tokens"].max(),
        "Std Dev": f"{df['reasoning_tokens'].std():.0f}"
    })

summary_df = pd.DataFrame(summary_stats)
print("\nSummary Statistics:")
print(summary_df.to_string(index=False))


Summary Statistics:
                                  Model  Total Samples Avg Tokens Median Tokens  Min Tokens  Max Tokens Std Dev
     RedHatAI-Qwen3-32B-quantized.w4a16             20        129           128          92         172      22
RedHatAI-gemma-3-27b-it-quantized.w4a16             20        108           104          63         170      29
